# Coverage comparison helper and plots

This notebook provides a reusable plotting helper to compare coverage over time windows for multiple pairs:
- Framework × FlashFuzz (ours, possibly with variants)
- Framework × baseline(s) (ACETest, TitanFuzz, PathFinder, etc.)

We display plots inline (no reference lines) and optionally save PNG/PDF into the top-level `plots/` folder.

In [9]:
# Imports and reusable plot helper

import os

import matplotlib

matplotlib.use("Agg")  # safe for headless save; inline display still works in notebook

import matplotlib.pyplot as plt

import matplotlib.ticker as mticker

# Make labels and fonts larger for readability
plt.rcParams.update({
    'font.size': 20,
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
})






def plot_pair(time_bins, ours_vals, baseline_vals, *,

              framework: str,

              ours_label: str,

              baseline_label: str,

              show: bool = True,

              save_dir: str | None = None,

              filename_stem: str | None = None,

              figsize=(8, 4.5), dpi=160,

              baseline_drop_bins: list[str] | None = None,

              minimal: bool = True,

              y_tick_step: int | float | None = 1000,

              y_min: int | float | None = 0,

              y_max: int | float | None = None,

              y_label: str | None = None):

    """

    Plot a coverage curve pair.



    Args:

        time_bins: list[str], like ["0-60", "60-120", ...]

        ours_vals: list[int], coverage for ours

        baseline_vals: list[int | None], coverage for baseline

        framework: e.g., "TensorFlow" or "PyTorch" (not shown when minimal=True)

        ours_label: e.g., "FlashFuzz (ours)" or variant

        baseline_label: e.g., "PathFinder (baseline)"

        show: display inline if True

        save_dir: directory to save PNG/PDF

        filename_stem: base name for saved files

        figsize, dpi: figure size and DPI

        baseline_drop_bins: optional list of bin labels to drop ONLY from the baseline curve; the line stays continuous.

        minimal: if True, remove title and axis labels and omit framework name from legend entries

        y_tick_step: set a uniform major tick interval for y-axis (e.g., 500). If None, use matplotlib default.

        y_min, y_max: y-axis limits. Use None to auto. Defaults to y_min=0, y_max=None.

        y_label: optional custom y-axis label (used when minimal=False). Defaults to "Code branch coverage".

    """

    ends = [int(b.split('-')[1]) for b in time_bins]

    x_seconds = ends

    xmax = max(ends) if ends else 0



    fig = plt.figure(figsize=figsize, dpi=dpi)

    ax = fig.add_subplot(111)



    # Legend labels (omit framework name when minimal)

    ours_leg = ours_label if minimal else f"{framework} – {ours_label}"

    base_leg = baseline_label if minimal else f"{framework} – {baseline_label}"



    # Ours plotted over full x

    ax.plot(x_seconds, ours_vals, marker="o", linewidth=2, color="#1f77b4", label=ours_leg)



    # Baseline: optionally drop some bins while keeping line continuous

    if baseline_drop_bins:

        drop_set = set(baseline_drop_bins)

        baseline_x = []

        baseline_y = []

        for b, x, y in zip(time_bins, x_seconds, baseline_vals):

            if b in drop_set:

                continue

            baseline_x.append(x)

            baseline_y.append(y)

        ax.plot(baseline_x, baseline_y, marker="s", linewidth=2, color="#d62728", label=base_leg)

    else:

        ax.plot(x_seconds, baseline_vals, marker="s", linewidth=2, color="#d62728", label=base_leg)



    # Minimal styling: optionally show axis labels

    if not minimal:

        ax.set_xlabel("Time (seconds)")

        ax.set_ylabel(y_label or "Code branch coverage")

        ax.set_title(f"{framework}: {ours_label} vs {baseline_label}")



    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.6)

    ax.legend(frameon=False, loc="lower right")



    step = 120 if xmax >= 600 else 60 if xmax >= 300 else 30

    ax.set_xticks(list(range(0, xmax + step, step)))

    ax.set_xlim(0, xmax)



    # Uniform y-axis interval and limits across plots

    ylim_kwargs = {}

    if y_min is not None:

        ylim_kwargs["bottom"] = y_min

    if y_max is not None:

        ylim_kwargs["top"] = y_max

    if ylim_kwargs:

        ax.set_ylim(**ylim_kwargs)

    else:

        ax.set_ylim(bottom=0)

    if y_tick_step is not None:

        ax.yaxis.set_major_locator(mticker.MultipleLocator(y_tick_step))



    fig.tight_layout()



    saved = {}

    if save_dir is not None:

        os.makedirs(save_dir, exist_ok=True)

        if filename_stem is None:

            safe_framework = framework.lower().replace(" ", "_")

            filename_stem = f"{safe_framework}_ours_vs_baseline"

        png_path = os.path.join(save_dir, f"{filename_stem}.png")

        pdf_path = os.path.join(save_dir, f"{filename_stem}.pdf")

        fig.savefig(png_path)

        fig.savefig(pdf_path)

        saved = {"png": png_path, "pdf": pdf_path}



    if show:

        plt.show()



    return {"fig": fig, "ax": ax, **saved}



print("plot_pair helper ready (minimal style by default).")

plot_pair helper ready (minimal style by default).


In [10]:
# Global plot config for uniform y-axes across figures

Y_TICK_STEP = 1000  # uniform step for y-axis ticks

Y_MIN = 0

Y_MAX_TORCH = 9000   # covers up to ~8457 in current PyTorch data

Y_MAX_TF = 14000     # raised to cover up to ~13749 in new TensorFlow data


In [11]:
# TensorFlow example: FlashFuzz (ours) vs PathFinder (baseline)

bins_tf = [
    "0-60", "60-120", "120-180", "180-240", "240-300", "300-360",
    "360-420", "420-480", "480-540", "540-600"
]

pathfinder_tf = [
    3319, 3692, 3905, 4097, 4339, 4507, 4641, 4741, 4809, 4867,
]

flashfuzz_tf = [
    9549, 9730, 9933, 10031, 10070, 10174, 10281, 10305, 10342, 10361,

]

_ = plot_pair(
    bins_tf, flashfuzz_tf, pathfinder_tf,
    framework="TensorFlow",
    ours_label="FlashFuzz",
    baseline_label="PathFinder",
    show=True,
    save_dir=os.path.abspath(os.path.join("..", "plots")),
    filename_stem="tensorflow_flashfuzz_vs_pathfinder",
    minimal=True,
    y_tick_step=Y_TICK_STEP,
    y_min=Y_MIN,
    y_max=Y_MAX_TF,
)
print("TensorFlow example plotted and saved.")

TensorFlow example plotted and saved.


In [12]:
# PyTorch: ACETest vs FlashFuzz (ours)

bins_10 = [
    "0-60", "60-120", "120-180", "180-240", "240-300",
    "300-360", "360-420", "420-480", "480-540", "540-600"
]

acetest_torch = [3679, 3768, 3784, 3828, 3837, 3842, 3854, 3863, 3871, 3877]
flashfuzz_torch_acetest = [4843, 5053, 5121, 5155, 5181, 5205, 5217, 5240, 5282, 5285]

_ = plot_pair(
    bins_10, flashfuzz_torch_acetest, acetest_torch,
    framework="PyTorch",
    ours_label="FlashFuzz",
    baseline_label="ACETest",
    show=True,
    save_dir=os.path.abspath(os.path.join("..", "plots")),
    filename_stem="pytorch_flashfuzz_vs_acetest",
    minimal=True,
    y_tick_step=Y_TICK_STEP,
    y_min=Y_MIN,
    y_max=Y_MAX_TORCH,
)
print("PyTorch ACETest vs FlashFuzz plotted and saved.")

PyTorch ACETest vs FlashFuzz plotted and saved.


In [13]:
# PyTorch: TitanFuzz vs FlashFuzz (ours) and FlashFuzz-all (ours)

bins_10 = [
    "0-60", "60-120", "120-180", "180-240", "240-300",
    "300-360", "360-420", "420-480", "480-540", "540-600"
]

titanfuzz_torch = [3816, 4351, 4546, 4652, 4803, 4945, 5016, 5165, 5212, 5241]
flashfuzz_torch_titan = [4867, 4943, 5000, 5089, 5103, 5111, 5118, 5142, 5155, 5162]
flashfuzz_all_torch = [7609, 7738, 7808, 7867, 7921, 7943, 7970, 7983, 8002, 8025]

_ = plot_pair(
    bins_10, flashfuzz_torch_titan, titanfuzz_torch,
    framework="PyTorch",
    ours_label="FlashFuzz",
    baseline_label="TitanFuzz",
    show=True,
    save_dir=os.path.abspath(os.path.join("..", "plots")),
    filename_stem="pytorch_flashfuzz_vs_titanfuzz",
    minimal=True,
    y_tick_step=Y_TICK_STEP,
    y_min=Y_MIN,
    y_max=Y_MAX_TORCH,
)

_ = plot_pair(
    bins_10, flashfuzz_all_torch, titanfuzz_torch,
    framework="PyTorch",
    ours_label="FlashFuzz",
    baseline_label="TitanFuzz",
    show=True,
    save_dir=os.path.abspath(os.path.join("..", "plots")),
    filename_stem="pytorch_flashfuzz_all_vs_titanfuzz",
    minimal=True,
    y_tick_step=Y_TICK_STEP,
    y_min=Y_MIN,
    y_max=Y_MAX_TORCH,
)

print("PyTorch TitanFuzz comparisons plotted (minimal style) and saved.")

PyTorch TitanFuzz comparisons plotted (minimal style) and saved.


In [14]:
# PyTorch: PathFinder vs FlashFuzz (ours)

bins_10 = [

    "0-60", "60-120", "120-180", "180-240", "240-300",

    "300-360", "360-420", "420-480", "480-540", "540-600"

]



# Baseline (PathFinder) for PyTorch — 10 bins

pathfinder_torch = [5090, 5276, 5375, 5441, 5478, 5504, 5514, 5522, 5523, 5537]



# Ours (FlashFuzz) for PyTorch — 10 bins

flashfuzz_torch_pathfinder = [8039, 8211, 8288, 8334, 8369, 8386, 8399, 8419, 8439, 8457]



_ = plot_pair(

    bins_10, flashfuzz_torch_pathfinder, pathfinder_torch,

    framework="PyTorch",

    ours_label="FlashFuzz",

    baseline_label="PathFinder",

    show=True,

    save_dir=os.path.abspath(os.path.join("..", "plots")),

    filename_stem="pytorch_flashfuzz_vs_pathfinder",


    minimal=True,

    y_tick_step=Y_TICK_STEP,

    y_min=Y_MIN,

    y_max=Y_MAX_TORCH,

)

print("PyTorch PathFinder vs FlashFuzz plotted and saved.")

PyTorch PathFinder vs FlashFuzz plotted and saved.


In [15]:
# TensorFlow: ACETest vs FlashFuzz (ours)

bins_10 = [

    "0-60", "60-120", "120-180", "180-240", "240-300",

    "300-360", "360-420", "420-480", "480-540", "540-600"

]



# Baseline (ACETest) for TensorFlow — 10 bins

acetest_tf = [8257, 9290, 11311, 11373, 11491, 11491, 11491, 11521, 11537, 11537]



# Ours (FlashFuzz) for TensorFlow — 10 bins

flashfuzz_tf_acetest = [11052, 11328, 11491, 11595, 11669, 11699, 11748, 11773, 11825, 11848]



_ = plot_pair(

    bins_10, flashfuzz_tf_acetest, acetest_tf,

    framework="TensorFlow",

    ours_label="FlashFuzz",

    baseline_label="ACETest",

    show=True,

    save_dir=os.path.abspath(os.path.join("..", "plots")),

    filename_stem="tensorflow_flashfuzz_vs_acetest",

    

    minimal=True,

    y_tick_step=Y_TICK_STEP,

    y_min=Y_MIN,

    y_max=Y_MAX_TF,

)

print("TensorFlow ACETest vs FlashFuzz plotted and saved.")

TensorFlow ACETest vs FlashFuzz plotted and saved.


In [16]:
# TensorFlow: TitanFuzz vs FlashFuzz (ours)

bins_10 = [

    "0-60", "60-120", "120-180", "180-240", "240-300",

    "300-360", "360-420", "420-480", "480-540", "540-600"

]



# Baseline (TitanFuzz) for TensorFlow — 10 bins

titanfuzz_tf = [12164, 12623, 12829, 12993, 13099, 13336, 13394, 13400, 13586, 13595]



# Ours (FlashFuzz) for TensorFlow — 10 bins

flashfuzz_tf_titan = [12821, 13163, 13361, 13462, 13548, 13617, 13649, 13694, 13714, 13749]



_ = plot_pair(

    bins_10, flashfuzz_tf_titan, titanfuzz_tf,

    framework="TensorFlow",

    ours_label="FlashFuzz",

    baseline_label="TitanFuzz",

    show=True,

    save_dir=os.path.abspath(os.path.join("..", "plots")),

    filename_stem="tensorflow_flashfuzz_vs_titanfuzz",

    minimal=True,

    y_tick_step=Y_TICK_STEP,

    y_min=Y_MIN,

    y_max=Y_MAX_TF,

)

print("TensorFlow TitanFuzz vs FlashFuzz plotted and saved.")

TensorFlow TitanFuzz vs FlashFuzz plotted and saved.
